# Functions on our 5 projects: CRG and Module, full output (fresh run)
Reuses the exact code of `treesitter-module.ipynb` (read from that notebook, not edited) for: CRG graph read, compiler references, module, and matching.
New here: every row each tool reported is classified, so we get **reported, missed, wrong** per tool.
- A tool row is **real** if a reference function in the same file has the same name and the row's line is inside it (same rule as 'found').
- A tool row is an **unnamed function** if it starts on the line of a function the reference lists as unnamed (e.g. a callback). Real, but has no name to match.
- Anything else is **wrong** (reported, but not a function per the compiler).
Output: `results/five-projects-functions/<stamp>/`.

In [1]:
import json, pathlib
SRC = [''.join(c['source']) for c in json.load(open('treesitter-module.ipynb'))['cells'] if c['cell_type'] == 'code']
SRC[1] = SRC[1].replace("RESULTS / 'treesitter-module' / f'{dt.datetime.now():%Y-%m-%d_%H%M}'", "RESULTS / 'five-projects-functions' / f'{dt.datetime.now():%Y-%m-%d_%H%M}'")
assert 'five-projects-functions' in SRC[1]
for i in range(0, 7):
    exec(SRC[i])
named_scores

$ node /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/js-ref/functions.mjs /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/crm-prototype app.js configurations/database.js configurations/env.config.js configurations/logger.config.js configurations/s3.config.js configurations/smtp.config.js conf


$ node /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/js-ref/functions.mjs /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/crm-prototype-client eslint.config.js src/App.jsx src/components/InviteModal.jsx src/components/LanguageToggle.jsx src/components/ProtectedRoute.jsx src/components/app-sid


$ /usr/bin/java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/java-ref/ListMethods.java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/spring-petclinic


$ /usr/bin/java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/java-ref/ListMethods.java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/guava


crm-prototype: 227 named, 63 unnamed (0s)
crm-prototype-client: 623 named, 438 unnamed (0s)
voice-assessment-agent: 487 named, 51 unnamed (0s)
spring-petclinic: 195 named, 21 unnamed (0s)


guava: 59236 named, 6153 unnamed (3s)


,app,language,tool,expected,found,missed,found %,lost to same name
0,crm-prototype,javascript,code-review-graph,227,206,21,90.7,0
1,crm-prototype,javascript,module,227,227,0,100.0,0
2,crm-prototype,javascript,both,227,227,0,100.0,0
3,crm-prototype-client,javascript,code-review-graph,623,559,64,89.7,0
4,crm-prototype-client,javascript,module,623,623,0,100.0,0
5,crm-prototype-client,javascript,both,623,623,0,100.0,0
6,voice-assessment-agent,python,code-review-graph,487,487,0,100.0,0
7,voice-assessment-agent,python,module,487,487,0,100.0,0
8,voice-assessment-agent,python,both,487,487,0,100.0,0
9,spring-petclinic,java,code-review-graph,195,192,3,98.5,3


In [2]:
from collections import Counter
rows, wrong_rows = [], []
for app in APPS:
    ref = expected[app]
    tools = {'CRG': crg[app]['funcs'][['file', 'name', 'line_start']], 'Module': mod[app][['file', 'name', 'line_start', 'named']]}
    for tool, df in tools.items():
        real = unnamed = wrong = 0
        for rel, g in df.groupby('file'):
            r = ref.get(rel, {'named': [], 'unnamed': []})
            free = Counter(s for s, _ in r['unnamed'])
            for x in g.itertuples():
                if any(n == x.name and s <= x.line_start <= e for n, s, e in r['named']):
                    real += 1
                elif free[x.line_start] > 0:
                    free[x.line_start] -= 1; unnamed += 1
                else:
                    wrong += 1; wrong_rows.append({'app': app, 'tool': tool, 'file': rel, 'name': x.name, 'line': x.line_start})
        m = matched[(app, 'code-review-graph' if tool == 'CRG' else 'module')]
        found = int((m.status == 'found').sum())
        rows.append({'project': app, 'tool': tool, 'real count': len(m), 'reported': len(df), 'matched named': real,
                     'unnamed functions': unnamed, 'wrong': wrong, 'missed': len(m) - found,
                     'coverage %': round(100 * found / len(m), 1), 'accuracy %': round(100 * (len(df) - wrong) / len(df), 1) if len(df) else None})
fn = pd.DataFrame(rows); wr = pd.DataFrame(wrong_rows)
fn.to_csv(OUT / 'functions_full.csv', index=False); wr.to_csv(OUT / 'wrong_rows.csv', index=False)
print(OUT); fn

/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/five-projects-functions/2026-11-20_1925


,project,tool,real count,reported,matched named,unnamed functions,wrong,missed,coverage %,accuracy %
0,crm-prototype,CRG,227,241,206,35,0,21,90.7,100.0
1,crm-prototype,Module,227,290,227,63,0,0,100.0,100.0
2,crm-prototype-client,CRG,623,563,559,3,1,64,89.7,99.8
3,crm-prototype-client,Module,623,1061,623,438,0,0,100.0,100.0
4,voice-assessment-agent,CRG,487,487,487,0,0,0,100.0,100.0
5,voice-assessment-agent,Module,487,538,487,51,0,0,100.0,100.0
6,spring-petclinic,CRG,195,192,192,0,0,3,98.5,100.0
7,spring-petclinic,Module,195,216,195,21,0,0,100.0,100.0
8,guava,CRG,59236,59188,59188,0,0,48,99.9,100.0
9,guava,Module,59236,65389,59236,6153,0,0,100.0,100.0


In [3]:
# examples of wrong rows per project and tool
wr.groupby(['app', 'tool']).head(5)

,app,tool,file,name,line
0,crm-prototype-client,CRG,src/store/userStore.jsx,sessionStorage,20
